# Phase 1 – Data preprocessing (corrected)

**Principle:** this notebook does only *stateless* cleaning (parsing, encoding, log transforms, extracting features from text).
Anything that *learns from the data* (imputation values, scaling, PCA, feature selection) is deliberately **left for the Phase 2 modelling `Pipeline`**, so it is fitted inside each cross-validation fold and cannot leak.

**What changed vs. the original notebook**

| # | Original issue | Fix |
|---|---|---|
| 1 | `fh` and `extracapsular_ext` were already 0/1; `.map({'Yes','No'})` turned them into all-zero columns | Kept as-is, validated to be {0,1} |
| 2 | Object columns (`bx`, `meds`, `mri_hospital`, ...) were silently ignored by `select_dtypes(number)` | `bx` encoded; every other text column is dropped *explicitly* with a reason; final assert: no object columns |
| 3 | `StandardScaler` / median / mode imputation fitted on all 195 rows | Removed – moved to Phase 2 pipeline |
| 4 | `lesion_max_mm` NaN filled with 0 (0 mm is not a real size); regex filled almost nothing | Left as NaN + missing flag. New `lesion_mm_report` parsed from the report (188/195 coverage) |
| 5 | ISUP regex missed "ISUP grade group 3" / "ISUP GG1" (2 hits instead of 35) | Regex fixed, Gleason fallback kept |
| 6 | `active_surveillance` matched "occupation **as**", "PSA surveillance"; `lesion_growth` matched "abnormal growths" | Strict pattern for surveillance; `lesion_growth` dropped (no valid mentions exist in the text) |
| 7 | `smoking_pack_years` NaN → 0 for everybody | 0 only for never-smokers; others stay NaN |
| 8 | `cspca` 0.0 sentinel was median-filled on the full data | Converted to NaN only; imputed later in the pipeline |
| 9 | Missing `pirads` filled with the mode (a 4/5) although the report says "no PI-RADS 3+ lesion" | Filled with 2 based on that report phrase (validated on labelled rows) |

In [1]:
import re, json
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100, 'display.width', 160)

DATA_DIR = '.'            # folder containing features.csv / embeddings_mri.csv
raw = pd.read_csv(f'{DATA_DIR}/features.csv')
df = raw.copy()

assert df['case_id'].is_unique
print('shape:', df.shape)
print('label counts (NaN = unlabeled):')
print(df['label'].value_counts(dropna=False))

shape: (195, 85)
label counts (NaN = unlabeled):
label
NaN    104
1.0     56
0.0     35
Name: count, dtype: int64


## 1. Features extracted from free text (must run *before* the text columns are dropped)

In [2]:
TEXT_COLS = ['note_history', 'previous_notes', 'radiology_report',
             'note_chief_complaint', 'note_physical_examination']
all_text = df[TEXT_COLS].fillna('').agg(' '.join, axis=1).str.lower()
report   = df['radiology_report'].fillna('').str.lower()

# ---- 1a. Lesion size from the radiology report -------------------------------------------
# Handles '12 mm', '12-mm', '1.2 cm'. Takes the largest mm/cm mention (volumes are in mL, so unaffected).
MM = re.compile(r'(\d+(?:\.\d+)?)\s*-?\s*(mm|cm)\b')
def lesion_mm_from_report(t):
    sizes = [float(v) * (10 if u == 'cm' else 1) for v, u in MM.findall(t)]
    return max(sizes) if sizes else np.nan

df['lesion_mm_report'] = report.apply(lesion_mm_from_report)

# Validation against the structured column. NOTE: the two measurements are NOT identical
# (structured value is systematically larger, e.g. 24 vs 20), so we keep them as SEPARATE features
# instead of mixing them in a single column like the original notebook did.
both = df[['lesion_max_mm', 'lesion_mm_report']].dropna()
print(f"lesion_mm_report coverage: {df['lesion_mm_report'].notna().sum()}/{len(df)}  "
      f"(lesion_max_mm: {df['lesion_max_mm'].notna().sum()}/{len(df)})")
print(f"Spearman corr on {len(both)} rows with both: {both.corr(method='spearman').iloc[0,1]:.2f}; "
      f"median ratio structured/report: {(both['lesion_max_mm']/both['lesion_mm_report']).median():.2f}")

# ---- 1b. Prior ISUP grade (Gleason fallback) ---------------------------------------------
ISUP = re.compile(r'isup\s*(?:grade\s*group\s*|grade\s*|gg\s*|group\s*)?([1-5])')
GLEASON = re.compile(r'gleason\s*(?:score\s*)?(\d)\s*\+\s*(\d)')
def prior_isup(t):
    m = ISUP.search(t)
    if m: return int(m.group(1))
    m = GLEASON.search(t)
    if m:
        a, b = int(m.group(1)), int(m.group(2)); s = a + b
        if s <= 6: return 1
        if (a, b) == (3, 4): return 2
        if (a, b) == (4, 3): return 3
        if s == 8: return 4
        if s >= 9: return 5
    return 0

df['prior_isup_grade'] = all_text.apply(prior_isup)
df['has_prior_isup']   = (df['prior_isup_grade'] > 0).astype(int)

# ---- 1c. Active surveillance (strict: 'on as' / bare 'surveillance' matched unrelated text) --
df['active_surveillance'] = all_text.str.contains(r'active surveillance|surveillance protocol').astype(int)

# ---- 1d. Medication flags from meds / pmhx / medhx ---------------------------------------
hx = (df['meds'].fillna('') + ' ' + df['pmhx'].fillna('') + ' ' + df['medhx'].fillna('')).str.lower()
KW = {
    'flag_5_ari':         ['finasteride', 'proscar', 'dutasteride', 'avodart', '5-ari', '5-alpha'],
    'flag_anticoagulant': ['warfarin', 'apixaban', 'eliquis', 'rivaroxaban', 'xarelto', 'dabigatran',
                           'pradaxa', 'heparin', 'anticoagulant', 'blood thinner'],
    'flag_alpha_blocker': ['tamsulosin', 'flomax', 'alfuzosin', 'uroxatral', 'doxazosin', 'cardura',
                           'terazosin', 'hytrin', 'alpha-blocker', 'alpha blocker'],
}
for col, words in KW.items():
    df[col] = hx.apply(lambda x: int(any(w in x for w in words)))

print(df[['prior_isup_grade', 'has_prior_isup', 'active_surveillance', *KW]].astype(float).sum().astype(int).to_string())

lesion_mm_report coverage: 188/195  (lesion_max_mm: 88/195)
Spearman corr on 88 rows with both: 0.77; median ratio structured/report: 1.21
prior_isup_grade       107
has_prior_isup          50
active_surveillance     10
flag_5_ari               0
flag_anticoagulant       4
flag_alpha_blocker      21


**Why `lesion_growth` is dropped:** I searched all five text columns for growth phrases ("increase in size", "interval growth", "larger/enlarged lesion", "previously measured"...).
There are 0 valid mentions – the original flag only fired on "abnormal growths" in a physical-exam sentence. A feature with no signal is removed rather than kept as noise.

## 2. Structured columns

In [3]:
# ---- 2a. Binary columns that are ALREADY 0/1 (original code destroyed them) ---------------
for c in ['fh', 'extracapsular_ext']:
    assert set(df[c].dropna().unique()) <= {0, 1}, c
print('fh:', df['fh'].value_counts().to_dict(), '| extracapsular_ext:', df['extracapsular_ext'].value_counts().to_dict())

# ---- 2b. DRE -> abnormal / not done (reference level = normal) ----------------------------
dre = df['dre'].astype(str).str.strip().str.lower()
df['dre_abnormal'] = dre.isin(['abnormal', 'nodus', 'suspicious']).astype(int)
df['dre_not_done'] = dre.isin(['not done', 'nan']).astype(int)

# ---- 2c. Prior biopsy result `bx` (Positive / Negative / None).  NaN means no prior biopsy ---
# !! Verify with the data owner that `bx` is a PRIOR biopsy and is known at decision time. !!
df['bx_cat'] = df['bx'].fillna('None')

# ---- 2d. Lesion zone: peripheral is the reference; central (n=2) + transition (n=9) are too rare to keep apart
df['zone_unknown']    = df['lesion_zone'].isna().astype(int)
df['zone_non_periph'] = df['lesion_zone'].isin(['transition', 'central']).astype(int)

# ---- 2e. One-hot for the remaining small categoricals ------------------------------------
df = pd.get_dummies(df, columns=['enc_type', 'bx_cat', 'smoking_status'], dtype=int)

# ---- 2f. Pack-years: 0 only for never-smokers; leave other NaNs for the Phase 2 imputer ----
never = df['smoking_status_never'] == 1
df.loc[never & df['smoking_pack_years'].isna(), 'smoking_pack_years'] = 0.0

# ---- 2g. PI-RADS: the single missing value ------------------------------------------------
# Reports without a visible lesion say "no PI-RADS 3+ lesion identified". Check that phrase <-> pirads == 2:
no_lesion = report.str.contains(r'no pi-?rads 3\+? lesion')
print(pd.crosstab(no_lesion, df['pirads'].fillna(-1)).rename(columns={-1: 'NaN'}))

fh: {0: 169, 1: 26} | extracapsular_ext: {0: 153, 1: 42}
pirads            NaN  2.0  3.0  4.0  5.0
radiology_report                         
False               0   10   18   72   90
True                1    4    0    0    0


In [4]:
assert (df.loc[no_lesion & df['pirads'].notna(), 'pirads'] == 2).all()   # phrase <-> PI-RADS 2 on all labelled-pirads rows
df.loc[no_lesion & df['pirads'].isna(), 'pirads'] = 2
print('remaining NaN pirads:', df['pirads'].isna().sum())

# ---- 2h. Missing-value policy: keep NaN, add an indicator only where missingness is large -----
df['lesion_max_mm_missing'] = df['lesion_max_mm'].isna().astype(int)   # 107/195 missing, NOT filled with 0
df['lab_hba1c_missing']     = df['lab_hba1c'].isna().astype(int)       # 151/195 missing (only tested in at-risk patients)

# ---- 2i. cspca: 0.0 is a sentinel for 'not computed' -> NaN (imputed later, inside CV) ------
print('cspca zeros -> NaN:', (df['cspca'] == 0).sum())
df['cspca'] = df['cspca'].replace(0.0, np.nan)

remaining NaN pirads: 0
cspca zeros -> NaN: 5


## 3. Skewed numeric variables: log transform only (scaling happens in the Phase 2 pipeline)

In [5]:
def slog1p(x):                      # sign-preserving log1p, safe for negative values (e.g. slopes)
    return np.sign(x) * np.log1p(np.abs(x))

SKEWED = ['psa', 'psa_first', 'psa_max', 'psad', 'psap', 'psav', 'psa_slope_per_step',
          'psa_last_over_first', 'vol']
for c in SKEWED:
    df[f'{c}_log'] = slog1p(df[c])
print(df[[f'{c}_log' for c in SKEWED]].describe().loc[['min', 'max']].round(2))

     psa_log  psa_first_log  psa_max_log  psad_log  psap_log  psav_log  psa_slope_per_step_log  psa_last_over_first_log  vol_log
min     0.18           0.18         0.34      0.01      0.34     -0.58                   -0.22                     0.69     2.86
max     5.25           2.79         5.25      1.63      5.04      3.75                    4.48                     3.74     5.13


## 4. Drop columns that are not features (every drop has a reason)

In [6]:
DROP = {
    # --- identical to the original notebook ---
    'enc_dept': 'constant (all Urology outpatient)', 'active_treatment_flag': 'constant (all 0)',
    'lab_egfr': 'dropped in original; creatinine kept (check if you want eGFR back)',
    'folder': 'duplicate identifier', 'enc_date': 'zero variance (2025)', 'admin': 'administrative metadata',
    'next_of_kin': 'personal metadata', 'enc_ref': 'GP name: 13 values / 91 labels -> spurious',
    'notes': 'template string built from other features (+cspca) -> redundant / leakage-prone',
    'family_history_raw': 'already in fh', 'alcohol': 'already in alcohol_units_week',
    'smoking': 'already in smoking_status / pack_years',
    'lab_psa': '~94% identical to psa', 'psa_last': '~99% identical to psa',
    # --- added: text columns that the original silently ignored ---
    'occupation': 'free text, no clinical signal, high cardinality', 'marital': 'social, high cardinality',
    'living': 'social, high cardinality', 'exercise': 'free text, high cardinality',
    'mri_hospital': 'site effect, ~12 sites for 91 labels -> spurious',
    'meds': 'converted to flag_* columns', 'pmhx': 'converted to flag_* / n_comorbidities',
    'medhx': 'duplicate of pmhx', 'psa_trend_values': 'raw list; summarised by slope / ratio / max',
    # --- raw columns replaced by processed versions ---
    'dre': 'replaced by dre_abnormal / dre_not_done', 'bx': 'replaced by bx_cat_*',
    'lesion_zone': 'replaced by zone_*',
    **{c: 'replaced by *_log version' for c in SKEWED},
    **{c: 'raw text, features extracted in section 1' for c in TEXT_COLS},
}
flag_cols = [c for c in df.columns if c.endswith('_flag')]            # lab H/L flags: redundant with lab values
all_null  = df.columns[df.isna().all()].tolist()                       # e.g. ct, lab_wbc_flag ...
to_drop = sorted(set(DROP) | set(flag_cols) | set(all_null))
missing_from_df = [c for c in to_drop if c not in df.columns]
assert not missing_from_df, missing_from_df
df = df.drop(columns=to_drop)
print('dropped', len(to_drop), 'columns; all-null:', all_null)
print('shape now:', df.shape)

dropped 52 columns; all-null: ['ct', 'lab_wbc_flag', 'lab_platelets_flag', 'lab_testosterone_flag', 'lab_alkaline_phosphatase_flag', 'lab_ldh_flag', 'lab_hba1c_flag']
shape now: (195, 62)


## 5. Final sanity checks

In [7]:
# 5a. Only case_id may be non-numeric -> nothing is silently ignored later
obj = df.select_dtypes(exclude=[np.number]).columns.tolist()
assert obj == ['case_id'], obj

# 5b. No infinities, no constant columns
feat_cols = [c for c in df.columns if c not in ('case_id', 'label')]
assert not np.isinf(df[feat_cols]).any().any()
const = [c for c in feat_cols if df[c].nunique(dropna=True) <= 1]
print('constant columns (dropped):', const)
df = df.drop(columns=const)
feat_cols = [c for c in feat_cols if c not in const]

# 5c. Binary flags that are too rare in the LABELLED set to learn anything (consider dropping in Phase 2)
lab = df[df['label'].notna()]
bin_cols = [c for c in feat_cols if set(lab[c].dropna().unique()) <= {0, 1}]
rare = {c: int(lab[c].sum()) for c in bin_cols if lab[c].sum() < 5 or (len(lab) - lab[c].sum()) < 5}
print('rare binary features in labelled data (n positives):', rare)

# 5d. Remaining NaNs -> to be imputed INSIDE the Phase 2 pipeline
na = df[feat_cols].isna().sum(); na = na[na > 0]
print('\nNaNs left for the pipeline imputer:\n', na.to_string())
print(f'\nFinal: {len(df)} rows ({lab.shape[0]} labelled) x {len(feat_cols)} features')

constant columns (dropped): ['flag_5_ari']
rare binary features in labelled data (n positives): {'flag_anticoagulant': 2, 'dre_not_done': 1, 'zone_unknown': 4}

NaNs left for the pipeline imputer:
 cspca                   5
smoking_pack_years      2
lesion_max_mm         107
lab_hba1c             151
lesion_mm_report        7

Final: 195 rows (91 labelled) x 59 features


## 6. Embeddings (stateless steps only: alignment + quality checks)

In [8]:
emb = pd.read_csv(f'{DATA_DIR}/embeddings_mri.csv')
emb_cols = [c for c in emb.columns if c.startswith('emb_')]
assert emb['case_id'].is_unique and not emb[emb_cols].isna().any().any()

print('embedding rows:', len(emb), '| dims:', len(emb_cols))
print('features rows without embedding:', (~df['case_id'].isin(emb['case_id'])).sum(),
      '| of which labelled:', (~lab['case_id'].isin(emb['case_id'])).sum())
print('embeddings with unknown case_id:', (~emb['case_id'].isin(df['case_id'])).sum())

# row-norm outliers (a very large norm = suspicious scan / failed extraction)
norms = np.linalg.norm(emb[emb_cols].values, axis=1)
z = (norms - norms.mean()) / norms.std()
print('rows with |z(norm)| > 3:', int((np.abs(z) > 3).sum()))
col_max = emb[emb_cols].abs().max().sort_values(ascending=False)
print('largest single values:\n', col_max.head(3).round(2).to_string())

# align to features order; flag patients without embedding (NaN rows are handled in Phase 2)
emb_aligned = df[['case_id']].merge(emb, on='case_id', how='left')
df['has_embedding'] = emb_aligned[emb_cols[0]].notna().astype(int).values

embedding rows: 191 | dims: 1024
features rows without embedding: 4 | of which labelled: 2
embeddings with unknown case_id: 0
rows with |z(norm)| > 3: 0


largest single values:
 emb_175    31.73
emb_687    27.37
emb_251    16.96


`has_embedding` is saved in the features file but is **only for bookkeeping** (do not feed it to the model).
Scaling, PCA (e.g. 5–20 components) and the choice to combine with tabular features happen in Phase 2, inside cross-validation.
`reasoning_audit.csv` is intentionally **not** used here – it records the post-decision reasoning and would leak the label.

## 7. Save

In [9]:
df.to_csv('features_clean.csv', index=False)
emb_aligned.to_csv('embeddings_aligned.csv', index=False)          # same row order as features_clean.csv, NaN rows where missing
json.dump({'features': [c for c in df.columns if c not in ('case_id', 'label', 'has_embedding')],
           'embedding_columns': emb_cols,
           'dropped_with_reason': DROP},
          open('feature_lists.json', 'w'), indent=2)
print('saved: features_clean.csv', df.shape, '| embeddings_aligned.csv', emb_aligned.shape, '| feature_lists.json')
df.head(3)

saved: features_clean.csv (195, 62) | embeddings_aligned.csv (195, 1025) | feature_lists.json


,case_id,label,age,months,cspca,pirads,ipss,alcohol_units_week,n_comorbidities,n_allergies,weight_kg,height_cm,bmi,bp_systolic,bp_diastolic,hr_bpm,smoking_pack_years,fh,lesion_max_mm,extracapsular_ext,n_previous_notes,psa_trend_n,lab_free_psa,lab_haemoglobin,lab_wbc,lab_platelets,lab_creatinine,lab_testosterone,lab_alkaline_phosphatase,lab_ldh,lab_hba1c,lesion_mm_report,prior_isup_grade,has_prior_isup,active_surveillance,flag_anticoagulant,flag_alpha_blocker,dre_abnormal,dre_not_done,zone_unknown,zone_non_periph,enc_type_Follow-up - PSA + MRI,enc_type_Re-evaluation - prior PCa diagnosis,enc_type_Re-evaluation - prior negative biopsy,bx_cat_Negative,bx_cat_None,bx_cat_Positive,smoking_status_current,smoking_status_ex,smoking_status_never,lesion_max_mm_missing,lab_hba1c_missing,psa_log,psa_first_log,psa_max_log,psad_log,psap_log,psav_log,psa_slope_per_step_log,psa_last_over_first_log,vol_log,has_embedding
0,PT-pseudo_0020cfca66c8,0.0,67,2,0.596203,2.0,18.0,27.0,0,0,68.0,175.0,22.2,147.0,77.0,79.0,22.0,1,NaN,0,2,4,26.0,16.1,8.9,265.0,94.0,14.1,82.0,176.0,NaN,9.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,1,1,1.740466,1.648659,1.740466,0.131028,1.686399,0.392042,-0.067659,0.750967,3.539799,1
1,PT-pseudo_0169468160c6,1.0,75,2,0.964853,5.0,1.0,8.0,3,0,83.0,158.0,33.2,149.0,99.0,75.0,10.0,0,24.0,1,1,3,19.0,14.8,4.6,285.0,108.0,9.9,73.0,146.0,5.5,20.0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,5.236442,2.595255,5.236442,1.449269,5.039547,3.670715,4.480740,2.777616,4.065259,1
2,PT-pseudo_01b643727538,0.0,71,1,0.852476,5.0,6.0,0.0,0,0,86.0,181.0,26.3,147.0,91.0,81.0,0.0,1,NaN,1,2,3,12.0,14.2,6.5,352.0,108.0,15.6,90.0,224.0,NaN,14.0,3,1,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1,1,1,2.332144,1.916923,2.332144,0.416075,2.261763,-0.343590,1.011601,0.956837,2.946017,1
